<a href="https://colab.research.google.com/github/NJ555/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NJ555/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from pathlib import Path
import pandas as pd

# Create folders
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("work/outputs").mkdir(parents=True, exist_ok=True)

# Download the dataset into Colab
csv_path = Path("data/raw/content_refresh_anonymized.csv")
if not csv_path.exists():
    !wget -q -O data/raw/content_refresh_anonymized.csv https://raw.githubusercontent.com/NJ555/flyrank-ml-starter/main/data/raw/content_refresh_anonymized.csv

# Confirm it loaded
df_check = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Rows:", len(df_check))
print("Columns:", list(df_check.columns))

Loaded: data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

My rule is: prioritize pages that still have traffic opportunity but are weak on CTR, weak on average position, or stale from not being updated for a while. The two main signals are CTR and average position, and freshness is used as a tie-breaker. Reason codes: low_ctr, poor_position, stale_content, high_opportunity.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_csv(filename="content_refresh_anonymized.csv"):
    search_roots = [Path("."), Path("/content"), Path("/content/drive/MyDrive")]
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {filename} in the current Colab/repo folders.")

data_path = find_csv()
df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)
print("Columns found:", list(df.columns)[:20])

# Make sure key numeric columns are numeric if they exist
numeric_cols = [
    "ctr",
    "avg_position",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

def score_row(row):
    score = 0

    impressions = row.get("impressions_90d", 0)
    ctr = row.get("ctr", 0)
    pos = row.get("avg_position", 0)
    stale = row.get("days_since_last_update", 0)

    # Opportunity signal
    if impressions >= 3000:
        score += 3
    elif impressions >= 300:
        score += 2
    elif impressions > 0:
        score += 1

    # CTR signal (ctr is a percentage, so 1 = 1%)
    if ctr < 1:
        score += 4
    elif ctr < 2:
        score += 2

    # Position signal (lower is better)
    if pos == 0:
        score += 1  # no data, keep visible
    elif pos > 50:
        score += 4
    elif pos > 20:
        score += 3
    elif pos > 10:
        score += 2

    # Freshness signal
    if stale >= 180:
        score += 3
    elif stale >= 90:
        score += 1

    return score

def reason_code(row):
    ctr = row.get("ctr", 0)
    pos = row.get("avg_position", 0)
    stale = row.get("days_since_last_update", 0)

    if ctr < 1 and pos > 20 and stale >= 180:
        return "low_ctr_poor_position_stale"
    if ctr < 1 and pos > 20:
        return "low_ctr_poor_position"
    if ctr < 2 and stale >= 180:
        return "low_ctr_stale"
    if pos > 20:
        return "poor_position"
    if stale >= 180:
        return "stale_content"
    return "general_review"

def action_label(row):
    score = row["score"]
    if score >= 8:
        return "review_first"
    elif score >= 5:
        return "review_next"
    else:
        return "monitor"

df["score"] = df.apply(score_row, axis=1)
df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df.apply(action_label, axis=1)

df = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

print(df[["rank", "score", "reason_code", "action_label"]].head(10))

Loaded: data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Columns found: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']
   rank  score                  reason_code  action_label
0     1     13  low_ctr_poor_position_stale  review_first
1     2     13  low_ctr_poor_position_stale  review_first
2     3     13  low_ctr_poor_position_stale  review_first
3     4     13  low_ctr_poor_position_stale  review_first
4     5     13  low_ctr_poor_position_stale  review_first
5     6     13  low_ctr_poor_position_stale  review_first
6     7     12        low_ctr_poor_position  review_first
7     8     12                low_ctr_stale  review_first
8     9     12        low_ctr_poor_position  review_first
9    10     12   

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

output_file = "work/outputs/baseline_action_score.csv"

df.to_csv(output_file, index=False)

print("CSV saved successfully.")
print(output_file)
print(df.head(10))

CSV saved successfully.
work/outputs/baseline_action_score.csv
             content_id          client_id  search_volume  competition  \
0  content_7368877ea310  client_7f2253d7e2            0.0         0.00   
1  content_1bfaa38ff26c  client_7f2253d7e2            0.0         0.00   
2  content_5feee3994adb  client_7f2253d7e2            0.0         0.00   
3  content_b16bd7307b39  client_7f2253d7e2            0.0         0.00   
4  content_ecb6215e79fd  client_7f2253d7e2            0.0         0.00   
5  content_6476d1d8c050  client_19581e27de           10.0         0.54   
6  content_109f8f7c9d39  client_6208ef0f77            0.0         0.00   
7  content_cf56e2e2e282  client_7f2253d7e2            0.0         0.00   
8  content_fb66dd8f4629  client_6208ef0f77            0.0         0.00   
9  content_62abc4bd66be  client_4e07408562         2900.0         0.14   

  competition_level   cpc     content_type    main_intent  word_count  \
0               LOW  0.00  keyword article  infor

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = [c for c in [
    "rank",
    "content_id",
    "score",
    "action_label",
    "reason_code"
] if c in df.columns]

print(df[cols].head(20))

    rank            content_id  score  action_label  \
0      1  content_7368877ea310     13  review_first   
1      2  content_1bfaa38ff26c     13  review_first   
2      3  content_5feee3994adb     13  review_first   
3      4  content_b16bd7307b39     13  review_first   
4      5  content_ecb6215e79fd     13  review_first   
5      6  content_6476d1d8c050     13  review_first   
6      7  content_109f8f7c9d39     12  review_first   
7      8  content_cf56e2e2e282     12  review_first   
8      9  content_fb66dd8f4629     12  review_first   
9     10  content_62abc4bd66be     12  review_first   
10    11  content_df71843dcd17     12  review_first   
11    12  content_d49c7fc84373     12  review_first   
12    13  content_9e5b6828d9db     12  review_first   
13    14  content_b45f91ebc732     12  review_first   
14    15  content_0a91db491d14     12  review_first   
15    16  content_41e96bf2e994     12  review_first   
16    17  content_e09b5602ba42     12  review_first   
17    18  

## 4. Weak picks + leakage check

The weak picks are the rows that only look interesting because of one signal, not because the whole pattern is strong. I also checked that no future-label columns were used in the score.

In [6]:
weak_picks = df.tail(10)

cols_to_show = [c for c in [
    "content_id",
    "rank",
    "score",
    "action_label",
    "reason_code",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "impressions_90d",
] if c in weak_picks.columns]

print(weak_picks[cols_to_show].to_string(index=False))

print("\nLeakage check:")
print("I did not use trend_direction or trend_pct in the score.")
print("I used only observed signals like ctr, avg_position, impressions_90d, clicks_90d, and days_since_last_update.")

          content_id  rank  score action_label    reason_code   ctr  avg_position  days_since_last_update  impressions_90d
content_841cd12bb30f 29991      1      monitor general_review  50.0           0.5                      20                2
content_166b50543317 29992      1      monitor general_review  50.0           4.0                      20                2
content_9c8b317e8e1f 29993      1      monitor general_review  50.0           3.0                      20                2
content_006b16e7a2e7 29994      1      monitor general_review 100.0           1.0                       8                1
content_bf398aa7400e 29995      1      monitor general_review 100.0           2.0                      20                1
content_98458bafe297 29996      1      monitor general_review 100.0           1.0                      20                1
content_4272d3a330a3 29997      1      monitor general_review 100.0           8.0                       8                1
content_a84e013a

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.